# Libraries

Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# Data Loading, Updation & Preprocessing

In [3]:
ROOT = Path.cwd()
DATA_DIR = ROOT / "Data"

TRAIN_FILE = DATA_DIR / "TEP_FaultFree_Training.csv"
VAL_FILE = DATA_DIR / "TEP_FaultFree_Validation.csv"
TEST_FILE = DATA_DIR / "TEP_Faulty_plus_FaultFree_Testing.csv"
ADJ_FILE = DATA_DIR / "Adjacency_Matrix.csv"

Training Dataset (No Fault)

In [4]:
df_FF_R = pd.read_csv(TRAIN_FILE)

float_cols = df_FF_R.select_dtypes(include="float").columns
df_FF_R[float_cols] = df_FF_R[float_cols].astype("float32")

Validation Dataset (No Fault)

In [5]:
df_FF_V = pd.read_csv(VAL_FILE)

float_cols = df_FF_V.select_dtypes(include="float").columns
df_FF_V[float_cols] = df_FF_V[float_cols].astype("float32")

Testing Dataset (Fault + No Fault)

In [6]:
df_T = pd.read_csv(TEST_FILE)

Preprocessing (Standard Normalization)

In [7]:
irrelevant_cols = ["faultNumber", "simulationRun", "sample"]
sensor_cols = [c for c in df_FF_R.columns if c not in irrelevant_cols]

mu_global = df_FF_R[sensor_cols].mean()
sigma_global = df_FF_R[sensor_cols].std() + 1e-6

df_FF_R[sensor_cols] = (df_FF_R[sensor_cols] - mu_global) / sigma_global
df_FF_V[sensor_cols] = (df_FF_V[sensor_cols] - mu_global) / sigma_global
df_T[sensor_cols] = (df_T[sensor_cols] - mu_global) / sigma_global

print("Training mean:", df_FF_R[sensor_cols].mean().mean())
print("Training std:", df_FF_R[sensor_cols].std().mean())

print("Validation mean:", df_FF_V[sensor_cols].mean().mean())
print("Validation std:", df_FF_V[sensor_cols].std().mean())

print("Test mean:", df_T[sensor_cols].mean().mean())
print("Test std:", df_T[sensor_cols].std().mean())

Training mean: -6.792694e-06
Training std: 0.99998766
Validation mean: 0.0037545073
Validation std: 1.0050281
Test mean: 0.20378595865659016
Test std: 5.479316098827725


# Adjacency matrix

Adjacency Matrix (1-52 shifted to 0-51)

In [8]:
A_matrix = pd.read_csv(ADJ_FILE)

A_matrix = A_matrix.drop(columns=["Unnamed: 0"])
A_matrix = A_matrix.iloc[:52]
A_matrix = A_matrix.apply(pd.to_numeric)
A_matrix.columns = range(len(A_matrix.columns))
A_matrix = A_matrix.astype("int64")

Conversion of A into edge list

In [9]:
A = torch.tensor(A_matrix.values, dtype=torch.long) # Converts your adjacency matrix A into a PyTorch tensor of integers
edge_index = A.nonzero(as_tuple=False).t().contiguous() # Convert adjacency to edge_index, transposed to get (2, num of edges) form, where each column is 1 edge

Feature vector function for node (d=1)

In [10]:
def get_X_t(df, t, d):
    x = df.iloc[t, 3:].values
    x = torch.tensor(x, dtype=torch.float32).view(52, d)
    return x

# Model Architecture for GAE

In [11]:
class GCNEncoder(nn.Module):
    def __init__(self, hidden_dim, latent_dim):
        super().__init__()
        self.conv1 = GCNConv(1, hidden_dim) # (Input -> Output) = (52xd -> 52xh)
        self.conv2 = GCNConv(hidden_dim, latent_dim) # (Input -> Output) = (52xh -> 52xl)

    def forward(self, x, edge_index): # For forward pass
        x = self.conv1(x, edge_index) # H(1) = A.X.W(0)
        x = F.relu(x)
        x = self.conv2(x, edge_index) # Z = A.H(1).W(1), Latent space representation
        return x

class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.linear = nn.Linear(latent_dim, 1) # (Input -> Output) = (52xl -> 52x1) , x_i = ​W.z_i​ + b

    def forward(self, z):
        return self.linear(z)

class GAE(nn.Module):
    def __init__(self, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = GCNEncoder(hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x, edge_index):
        z = self.encoder(x, edge_index)
        x_hat = self.decoder(z)
        return x_hat

# Model Initialization & Training

Variables to define

In [12]:
d = 1
hidden_dim = 16
latent_dim = 4

Model Initialization

In [13]:
model = GAE(hidden_dim, latent_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

Training & Validation Phase

In [14]:
best_val_loss = float('inf')
patience = 10
counter = 0

num_epochs = 50

for epoch in range(num_epochs):

    # Training
    model.train()
    train_loss = 0.0 # Accumulates total loss over all time steps t

    for t in range(len(df_FF_R)):
        X_t = get_X_t(df_FF_R, t, d) # Extracts node feature matrix for time t, X_t = (52,d)
                                     # Graph structure stays fixed

        optimizer.zero_grad() # Clears gradients from the previous step; if not done gradients explode as PyTorch accumulates gradients by default

        X_hat = model(X_t, edge_index) # Forward Pass of GAE model

        loss = loss_fn(X_hat, X_t) # Compares reconstructed node features with original ones; MSE loss

        loss.backward() # Computes gradients, i.e does BPTT

        optimizer.step() # Updates model weights using gradients
                         # Adam adjusts learning rate adaptively -> Adam does NOT change the global learning rate you set;
                         #                                       -> Instead, it computes a different effective learning rate for every parameter, based on past gradients.

        train_loss += loss.item() # Adds loss for this time step to epoch total

        train_loss /= len(df_FF_R)

    # Validation
    model.eval()
    val_loss = 0.0

    with torch.no_grad(): # Saves memory by not storing gradient values, which is calculated later on

        for t in range(len(df_FF_V)):

            X_t = get_X_t(df_FF_V, t, d)

            X_hat = model(X_t, edge_index)

            val_loss += loss_fn(X_hat, X_t).item()

    val_loss /= len(df_FF_V)

    print(f"Epoch {epoch}: Train {train_loss:.5f}, Val {val_loss:.5f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

Epoch 0: Train 0.00002, Val 0.92707
Epoch 1: Train 0.00002, Val 0.92708
Epoch 2: Train 0.00002, Val 0.92710
Epoch 3: Train 0.00002, Val 0.92713
Epoch 4: Train 0.00002, Val 0.92715
Epoch 5: Train 0.00002, Val 0.92717
Epoch 6: Train 0.00002, Val 0.92718
Epoch 7: Train 0.00002, Val 0.92719
Epoch 8: Train 0.00002, Val 0.92721
Epoch 9: Train 0.00002, Val 0.92722
Epoch 10: Train 0.00002, Val 0.92725
Early stopping triggered


# GAE vs GAE + KF vs GAE + RKF — FDR, FAR & per-fault detection

| Variant | Signal that is thresholded | Control limit |
|---|---|---|
| **GAE** | sample-wise reconstruction MSE (1 value per time step) | **one common threshold** |
| **GAE + KF** | Kalman-filtered per-node residuals (52 signals) | **individual threshold per node** |
| **GAE + RKF** | Hampel-robust Kalman-filtered per-node residuals (52 signals) | **individual threshold per node** |

**Needs these objects from the training notebook (same kernel):** `model`, `df_T`, `df_FF_V`, `edge_index`, `d`, `get_X_t`.

## 0. Configuration

In [15]:
# ── Detection settings ──────────────────────────────────────────
percentile  = 99      # control-limit percentile
persistence = 1       # consecutive alarms needed (1 = no persistence filter)
k_nodes     = 2       # KF / RKF: min. number of nodes above their own threshold to raise an alarm

# ── Kalman filter (KF) ──────────────────────────────────────────
KF_Q, KF_R = 0.001, 10.0

# ── Robust Kalman filter (RKF) ──────────────────────────────────
RKF_Q, RKF_R  = 0.01, 1.0
HAMPEL_SCALES = (2, 4, 6)     # (a, b, c) as multiples of the MAD-based sigma (estimated on df_FF_V)

# ── Where the control limits are learned from ───────────────────
#   "test_normal" : fault-free samples (faultNumber == 0) of the test set   -> original notebook behaviour
#   "validation"  : the fault-free validation set df_FF_V                    -> FAR is measured out-of-sample
THRESHOLD_SOURCE = "test_normal"

# ── Sanity check: objects expected from the training notebook ──
_missing = [n for n in ("model", "df_T", "df_FF_V", "edge_index", "d", "get_X_t") if n not in globals()]
assert not _missing, f"Missing from the kernel: {_missing}. Run the training notebook first."
assert THRESHOLD_SOURCE in ("test_normal", "validation")
## 1. Filters (KF and Hampel-robust KF)
import numpy as np
import pandas as pd
import torch


# ───────────────────────── Standard Kalman filter ─────────────────────────
def kalman_filter_errors(all_errors, Q=0.001, R=10.0, P0=1000.0):
    """Per-variable random-walk Kalman filter.  all_errors: (T, n_vars)"""
    T, n_vars = all_errors.shape

    kf_errors = np.zeros((T, n_vars))
    kf_errors[0, :] = all_errors[0, :]

    P = np.ones(n_vars) * P0
    Q = np.ones(n_vars) * Q if np.isscalar(Q) else np.asarray(Q)
    R = np.ones(n_vars) * R if np.isscalar(R) else np.asarray(R)

    for t in range(1, T):
        x_pred = kf_errors[t - 1, :]                         # prediction
        P_pred = P + Q
        K = P_pred / (P_pred + R)                            # Kalman gain
        kf_errors[t, :] = x_pred + K * (all_errors[t, :] - x_pred)   # update
        P = (1 - K) * P_pred                                 # covariance update

    return kf_errors


# ───────────────────────── Hampel influence function ──────────────────────
def hampel_psi(r, a, b, c):
    """psi(r): |r|<=a -> r | a<|r|<=b -> a*sign(r) | b<|r|<=c -> tapers to 0 | |r|>c -> 0"""
    abs_r, sign_r = np.abs(r), np.sign(r)
    psi = np.zeros_like(r, dtype=np.float64)

    z1 = abs_r <= a
    z2 = (abs_r > a) & (abs_r <= b)
    z3 = (abs_r > b) & (abs_r <= c)

    psi[z1] = r[z1]
    psi[z2] = a[z2] * sign_r[z2]
    psi[z3] = a[z3] * sign_r[z3] * (c[z3] - abs_r[z3]) / (c[z3] - b[z3] + 1e-8)
    return psi


def hampel_weight(r, a, b, c):
    """w = psi(r) / r in [0, 1]"""
    abs_r = np.abs(r)
    w = np.ones_like(r, dtype=np.float64)

    z2 = (abs_r > a) & (abs_r <= b)
    z3 = (abs_r > b) & (abs_r <= c)
    z4 = abs_r > c

    w[z2] = a[z2] / (abs_r[z2] + 1e-8)
    w[z3] = a[z3] * (c[z3] - abs_r[z3]) / ((abs_r[z3] + 1e-8) * (c[z3] - b[z3] + 1e-8))
    w[z4] = 0.0
    return np.clip(w, 0.0, 1.0)


def estimate_hampel_thresholds(normal_errors, scale_a=2.0, scale_b=4.0, scale_c=6.0):
    """Per-node (a, b, c) from fault-free errors; sigma_hat = 1.4826 * MAD."""
    median_val = np.median(normal_errors, axis=0)
    mad = np.median(np.abs(normal_errors - median_val), axis=0)
    sigma_hat = 1.4826 * mad

    eps = 1e-6
    a = np.maximum(scale_a * sigma_hat, eps)
    b = np.maximum(scale_b * sigma_hat, 2 * eps)
    c = np.maximum(scale_c * sigma_hat, 3 * eps)
    return a, b, c


# ───────────────────────── Robust Kalman filter ───────────────────────────
def robust_kalman_filter_hampel(all_errors, a, b, c, Q=0.01, R=1.0, P0=1000.0):
    """Kalman filter whose innovations pass through the Hampel psi function.
    Returns (filtered (T, n), weights (T, n))."""
    T, n = all_errors.shape
    Q = np.ones(n) * Q if np.isscalar(Q) else np.asarray(Q)
    R = np.ones(n) * R if np.isscalar(R) else np.asarray(R)

    x = all_errors[0].copy()
    P = np.ones(n) * P0

    kf_errors = np.zeros((T, n))
    kf_errors[0] = x
    weights = np.ones((T, n))

    for t in range(1, T):
        x_pred = x.copy()
        P_pred = P + Q

        innovation = all_errors[t] - x_pred
        psi_innov = hampel_psi(innovation, a, b, c)
        w_t = hampel_weight(innovation, a, b, c)

        R_eff = R / (w_t + 1e-8)                 # inflate noise where the sample is down-weighted
        K = P_pred / (P_pred + R_eff)

        x = x_pred + K * psi_innov
        P = (1 - K) * P_pred

        kf_errors[t] = x
        weights[t] = w_t

    return kf_errors, weights

## 2. Error computation and metric helpers

In [16]:
def compute_errors(df):
    """One pass over a dataframe (t starts at d, as in the original loops).

    Returns
    -------
    node_err   : (T, 52)  per-node residual ||x_i - x_hat_i||
    sample_err : (T,)     sample-wise MSE  (used by the common-threshold GAE)
    labels     : (T,)     faultNumber per time step
    """
    model.eval()
    node_err, sample_err, labels = [], [], []

    with torch.no_grad():
        for t in range(d, len(df)):
            X_t = get_X_t(df, t, d)
            X_hat = model(X_t, edge_index)

            node_err.append(torch.norm(X_t - X_hat, dim=1).cpu().numpy())
            sample_err.append(torch.mean((X_t - X_hat) ** 2).item())
            labels.append(df.loc[t, "faultNumber"])

    return np.array(node_err), np.array(sample_err), np.array(labels)


def apply_persistence(pred, window):
    """Alarm only if the last `window` samples were all flagged."""
    pred = np.asarray(pred).astype(int)
    if window <= 1:
        return pred
    out = np.zeros_like(pred)
    for i in range(len(pred)):
        if pred[max(0, i - window + 1): i + 1].sum() >= window:
            out[i] = 1
    return out


def evaluate(pred, labels):
    """Overall FDR / FAR and per-fault detection rates."""
    pred = np.asarray(pred).astype(int)
    true = (labels != 0).astype(int)

    TP = int(np.sum((pred == 1) & (true == 1)))
    FP = int(np.sum((pred == 1) & (true == 0)))
    TN = int(np.sum((pred == 0) & (true == 0)))
    FN = int(np.sum((pred == 0) & (true == 1)))

    FDR = TP / (TP + FN) if (TP + FN) > 0 else 0.0     # detection rate over all faulty samples
    FAR = FP / (FP + TN) if (FP + TN) > 0 else 0.0     # false alarms over fault-free samples

    per_fault = {int(f): float(pred[labels == f].mean()) for f in np.unique(labels) if f != 0}

    return dict(TP=TP, FP=FP, TN=TN, FN=FN, FDR=FDR, FAR=FAR, per_fault=per_fault)


## 3. Compute reconstruction errors (once)

In [17]:
# Test set (fault-free + faulty) and the fault-free validation set
node_err,     sample_err,     labels = compute_errors(df_T)
val_node_err, val_sample_err, _      = compute_errors(df_FF_V)

print("Test       :", node_err.shape, "| fault-free samples:", int((labels == 0).sum()))
print("Validation :", val_node_err.shape)
print("Threshold source:", THRESHOLD_SOURCE)

results = {}

Test       : (60479, 52) | fault-free samples: 2879
Validation : (9999, 52)
Threshold source: test_normal


## 4. GAE — one common threshold on the sample-wise error

In [18]:
ref = sample_err[labels == 0] if THRESHOLD_SOURCE == "test_normal" else val_sample_err

tau_gae = np.percentile(ref, percentile)                    # a single scalar control limit
pred_gae = apply_persistence(sample_err > tau_gae, persistence)

results["GAE"] = evaluate(pred_gae, labels)
print(f"GAE common threshold (p{percentile}): {tau_gae:.6f}")

GAE common threshold (p99): 1.810963


## 5. GAE + KF — individual threshold per node

In [19]:
kf_err = kalman_filter_errors(node_err, Q=KF_Q, R=KF_R)     # (T, 52)

if THRESHOLD_SOURCE == "test_normal":
    ref = kf_err[labels == 0]
else:
    ref = kalman_filter_errors(val_node_err, Q=KF_Q, R=KF_R)

tau_kf = np.percentile(ref, percentile, axis=0)             # (52,) one control limit per node
node_flags = kf_err > tau_kf
pred_kf = apply_persistence(node_flags.sum(axis=1) >= k_nodes, persistence)

results["GAE+KF"] = evaluate(pred_kf, labels)
print(f"GAE+KF thresholds: {tau_kf.shape[0]} per-node limits "
      f"(min {tau_kf.min():.4f}, median {np.median(tau_kf):.4f}, max {tau_kf.max():.4f})")

GAE+KF thresholds: 52 per-node limits (min 0.7812, median 0.9634, max 3.3813)


## 6. GAE + RKF — individual threshold per node

In [20]:
# Hampel zones (a, b, c) per node, estimated from fault-free validation errors
a, b, c = estimate_hampel_thresholds(val_node_err, *HAMPEL_SCALES)

rkf_err, rkf_weights = robust_kalman_filter_hampel(node_err, a, b, c, Q=RKF_Q, R=RKF_R)

if THRESHOLD_SOURCE == "test_normal":
    ref = rkf_err[labels == 0]
else:
    ref, _ = robust_kalman_filter_hampel(val_node_err, a, b, c, Q=RKF_Q, R=RKF_R)

tau_rkf = np.percentile(ref, percentile, axis=0)            # (52,) one control limit per node
node_flags = rkf_err > tau_rkf
pred_rkf = apply_persistence(node_flags.sum(axis=1) >= k_nodes, persistence)

results["GAE+RKF"] = evaluate(pred_rkf, labels)
print(f"GAE+RKF thresholds: {tau_rkf.shape[0]} per-node limits "
      f"(min {tau_rkf.min():.4f}, median {np.median(tau_rkf):.4f}, max {tau_rkf.max():.4f})")

GAE+RKF thresholds: 52 per-node limits (min 0.8889, median 1.3161, max 33.9705)


## 7. Comparison — FAR, FDR and per-fault detection rates

In [21]:
names = list(results)
threshold_type = {"GAE": "common (1 scalar)", "GAE+KF": "individual (52 per-node)", "GAE+RKF": "individual (52 per-node)"}

line = "=" * (14 + 26 * len(names))
print(line)
print(f"Percentile={percentile} | persistence={persistence} | k_nodes={k_nodes} | threshold source={THRESHOLD_SOURCE}")
print(line)
print(f"{'':<14}" + "".join(f"{n:>26}" for n in names))
print(f"{'Threshold':<14}" + "".join(f"{threshold_type[n]:>26}" for n in names))
print(f"{'FAR':<14}" + "".join(f"{results[n]['FAR']:>26.4f}" for n in names))
print(f"{'FDR (overall)':<14}" + "".join(f"{results[n]['FDR']:>26.4f}" for n in names))
print(f"{'TP / FP':<14}" + "".join(f"{str(results[n]['TP']) + ' / ' + str(results[n]['FP']):>26}" for n in names))
print(f"{'TN / FN':<14}" + "".join(f"{str(results[n]['TN']) + ' / ' + str(results[n]['FN']):>26}" for n in names))
print(line)

per_fault_df = pd.DataFrame({n: results[n]["per_fault"] for n in names})
per_fault_df.index.name = "Fault"
print("\nPer-fault detection rate")
print(per_fault_df.round(3).to_string())

Percentile=99 | persistence=1 | k_nodes=2 | threshold source=test_normal
                                     GAE                    GAE+KF                   GAE+RKF
Threshold              common (1 scalar)  individual (52 per-node)  individual (52 per-node)
FAR                               0.0101                    0.1042                    0.0959
FDR (overall)                     0.4764                    0.8280                    0.9364
TP / FP                       27440 / 29               47691 / 300               53939 / 276
TN / FN                     2850 / 30160               2579 / 9909               2603 / 3661

Per-fault detection rate
         GAE  GAE+KF  GAE+RKF
Fault                        
1      0.831   0.837    0.834
2      0.821   1.000    0.952
3      0.010   0.556    0.872
4      0.338   0.475    0.799
5      0.223   0.630    0.654
6      0.834   0.878    0.857
7      0.834   1.000    1.000
8      0.813   0.965    1.000
9      0.009   0.466    1.000
10     0.181 